In [1]:
import os 
import sys
import numpy as np
import pandas as pd
from pathlib import Path
sys.path.insert(0, "/home/s2864332/MySYNGAP")
# import artifactdetection as ad

PROJECT_ROOT = Path("/home/s2864332/MySYNGAP/MySYNGAP")
DIAGNOSE_ROOT = PROJECT_ROOT / "DiagnoseSYNGAP"
AD_ROOT = PROJECT_ROOT / "ArtifactDetection"
sys.path.insert(0, str(PROJECT_ROOT))                      
sys.path.insert(0, str(AD_ROOT))                           # for artifactdetection
sys.path.insert(0, str(DIAGNOSE_ROOT))                     # for DiagnoseSYNGAP

print("PYTHONPATH:")
for p in sys.path[:3]:
    print(" ", p)

print("Paths added:")
print(AD_ROOT)
print(DIAGNOSE_ROOT)

from ArtifactDetection.artifactdetection.load import LoadFiles
from ArtifactDetection.artifactdetection.implement import two_files, one_file

PYTHONPATH:
  /home/s2864332/MySYNGAP/MySYNGAP/DiagnoseSYNGAP
  /home/s2864332/MySYNGAP/MySYNGAP/ArtifactDetection
  /home/s2864332/MySYNGAP/MySYNGAP
Paths added:
/home/s2864332/MySYNGAP/MySYNGAP/ArtifactDetection
/home/s2864332/MySYNGAP/MySYNGAP/DiagnoseSYNGAP


In [2]:
path = '/exports/eddie/scratch/s2864332/SYNGAP_Rat_Data/formatted_raw/numpyformat_baseline'
count_npy = 0
count_pkl = 0
count_all = 0
for file in os.listdir(path):
    count_all += 1
    if file.endswith('.npy'):
        count_npy += 1
    elif file.endswith('.pkl'):
        count_pkl += 1


print(f"Number of .npy files: {count_npy}")
print(f"Number of .pkl files: {count_pkl}")
print(f"Total number of files: {count_all}")
        

Number of .npy files: 20
Number of .pkl files: 32
Total number of files: 52


### This notebook shows how to run Power calculations (following on from the Preprocessing notebook)

**To run the power analysis, you need to input the following:**
1. Path to .npy recording files and .pkl brain state files (they should be in the same folder) 
2. Path to save power analysis 
3. Which channel idx to run analysis on (chan_idx)
4. Dictionary of start and end times for each recording 
5. Overall list of animal ids 
6. If some recordings have two brain state files for each recording and others only have one, separate those into two separate lists

**An example is shown below with the described inputs**

### Required inputs

0. Preparing the metadata excell file

In [3]:
xlsx_path = '/home/s2864332/MySYNGAP/MySYNGAP/Recordings Samp Start-End TAINI - ALL DAYS.xlsx'

metadata = pd.read_excel(xlsx_path, header=None)
metadata = metadata.iloc[6:48, 2:8] # Adjusted to match the actual data range
metadata.columns = ['Animal ID', 'GENO', 'FILE NAME','DAY', 'Start Sample', 'End Sample'] # Assign column names
metadata["Animal ID"] = metadata["Animal ID"].ffill() # Fill down Animal ID
metadata["GENO"] = metadata["GENO"].ffill() # Fill down genotype

metadata.reset_index(drop=True, inplace=True) 
# metadata = metadata.drop("DAY", axis=1) # Drop DAY column

1. Path to .npy recordings and .pkl files 

In [4]:
directory_path = '/exports/eddie/scratch/s2864332/SYNGAP_Rat_Data/formatted_raw/numpyformat_baseline/'

2. Path to where power analysis should be saved 

In [5]:
save_folder = '/exports/eddie/scratch/s2864332/SYNGAP_Rat_Data/ArtifactDetectTest/power/'

3. Channel idx to select out, write a for loop to calculate more than one channel idx

In [6]:
chan_idx = 2

4. Dictionary of start and end times for each recording

In [7]:
SYNGAP_baseline_start = {'S7063_1': 15324481, 'S7063_2': 36959041, 
                       'S7064_1': 15324481, 'S7064_2':36959041,
                       'S7068_1': 12214513,
                       'S7069_1': 12214513, 'S7069_2':33849073,
                       'S7070_1': 16481329, 'S7070_2':38115889,
                       'S7071_1': 16481329, 'S7071_2':38115888,
                       'S7072_1': 16481329, 'S7072_2': 38115889,
                       'S7074_1': 35862289,
                       'S7075_1': 14227729,
                       'S7076_1': 17578081,
                       'S7083_1': 17578081, 'S7083_2':39212641,
                       'S7086_1': 12830497, 'S7086_2': 34465057,
                       'S7087_1': 39723457,
                       'S7088_1': 18088897, 
                       'S7091_1': 15369553, 'S7091_2': 37004113,
                       'S7092_1': 15369553,
                       'S7094_1': 34465057, 
                       'S7096_1': 34465057, 'S7096_2':56054794,
                       'S7098_1': 35246305, 'S7098_2':56880865,
                       'S7101_1': 35246305, 'S7101_2':56880864}


SYNGAP_baseline_end = {'S7063_1A': 36959040, 'S7063_2A': 58593600, 
                       'S7064_1A': 36959040, 'S7064_2A': 58593600,
                       'S7068_1A': 33849072,
                       'S7069_1A': 33849072, 'S7069_2A': 55483632,
                       'S7070_1A': 38115888, 'S7070_2A': 59750448,
                       'S7071_1A': 38115888, 'S7071_2A': 59750448,
                       'S7072_1A': 38115888, 'S7072_2A': 59750448,
                       'S7074_1A': 57496848,
                       'S7075_1A': 35862288,
                       'S7076_1A': 39212640,
                       'S7083_1A': 39212640, 'S7083_2A': 60847200,
                       'S7086_1A': 34465056, 'S7086_2A': 56099616,
                       'S7087_1A': 61358016,
                       'S7088_1A': 39723456, 
                       'S7091_1A': 37004112, 'S7091_2A': 58638672,
                       'S7092_1A': 37004112,
                       'S7094_1A': 56099616,
                       'S7096_1A': 56099616, 'S7096_2A': 77689353,
                       'S7098_1A': 56880864, 'S7098_2A': 78515425,
                       'S7101_1A': 56880864, 'S7101_2A': 78515424}

5. & 6. List of all animals, and those which have two recordings or only one

In [8]:
analysis_ls = []
SYNGAP_2_ls = []
SYNGAP_1_ls = []
for animal_id, group in metadata.groupby("Animal ID"):
    analysis_ls.append(animal_id)
    if group["DAY"].iloc[0] == 'NO' or group["DAY"].iloc[1] == 'NO':
        SYNGAP_1_ls.append(animal_id)
    else:
        SYNGAP_2_ls.append(animal_id)
        



In [9]:
print(150*"=")
print(f"Animals to process: {len(analysis_ls)}")
print(analysis_ls)
print(150*"=")
print(f"Animals with 2 recording: {len(SYNGAP_2_ls)}")
print(SYNGAP_2_ls)
print(150*"=")
print(f"Animals with 1 recordings: {len(SYNGAP_1_ls)}")
print(SYNGAP_1_ls)
print(150*"=")

Animals to process: 20
['S7063', 'S7064', 'S7068', 'S7069', 'S7070', 'S7071', 'S7072', 'S7074', 'S7075', 'S7076', 'S7083', 'S7086', 'S7087', 'S7088', 'S7091', 'S7092', 'S7094', 'S7096', 'S7098', 'S7101']
Animals with 2 recording: 12
['S7063', 'S7064', 'S7069', 'S7070', 'S7071', 'S7072', 'S7083', 'S7086', 'S7091', 'S7096', 'S7098', 'S7101']
Animals with 1 recordings: 8
['S7068', 'S7074', 'S7075', 'S7076', 'S7087', 'S7088', 'S7092', 'S7094']


### Run power analysis

In [10]:
for animal in analysis_ls:
    print(30*'=')
    print(animal)
    load_files = LoadFiles(directory_path = directory_path, animal_id = animal)
    if animal in SYNGAP_2_ls:
        num_epochs = 34560
        two_files(load_files = load_files, animal = animal, num_epochs = num_epochs, chan_idx = chan_idx,
                  save_folder = save_folder, start_times_dict = SYNGAP_baseline_start, end_times_dict = SYNGAP_baseline_end)
    if animal in SYNGAP_1_ls:
        num_epochs = 17280

        one_file(load_files = load_files, animal = animal, num_epochs = num_epochs, chan_idx = chan_idx,
                save_folder = save_folder, start_times_dict = SYNGAP_baseline_start, end_times_dict = SYNGAP_baseline_end)

S7063
S7064
S7068
S7069
S7070
S7071
S7072
S7074
S7075
S7076
S7083
S7086
S7087
S7088
S7091
S7092
S7094
S7096


/exports/csce/eddie/eng/groups/engbdat/Nikoo/conda/envs/DiagnoseSYNGAP/lib/python3.10/site-packages/scipy/signal/_spectral_py.py:2014: UserWarning: nperseg = 1252 is greater than input length  = 793, using nperseg = 793
  warnings.warn('nperseg = {0:d} is greater than input length '
/exports/csce/eddie/eng/groups/engbdat/Nikoo/conda/envs/DiagnoseSYNGAP/lib/python3.10/site-packages/scipy/signal/_spectral_py.py:2014: UserWarning: nperseg = 1252 is greater than input length  = 792, using nperseg = 792
  warnings.warn('nperseg = {0:d} is greater than input length '


S7098
S7101
